In [0]:
dim_customers = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.dim_customers")
dim_invoices = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.dim_invoices")
dim_products = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.dim_products")
fact_invoice_line_items = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.fact_invoice_line_items")
fact_payments = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.fact_payments")

In [0]:
display(fact_invoice_line_items)

In [0]:
# Total Revenue – Sum of all invoice line-item amounts
total_revenue = fact_invoice_line_items.agg({"usd_revenue": "sum"})
display(total_revenue)

In [0]:
#Total Quantity Sold – Sum of quantities sold across all products
total_quantity_sold = fact_invoice_line_items.agg({"quantity": "sum"})
display(total_quantity_sold)

In [0]:
# Average Invoice Value – Average total value per invoice
invoice_totals = fact_invoice_line_items.groupBy("invoice_id").agg({"usd_revenue": "sum"})
average_invoice_value = invoice_totals.agg({"sum(usd_revenue)": "avg"})
display(average_invoice_value)

In [0]:
# Top Customers by Revenue – Identify customers contributing the most revenue
fact_with_customer = fact_invoice_line_items.join(dim_invoices.select("invoice_id", "customer_id"), "invoice_id")
customer_revenue = fact_with_customer.groupBy("customer_id").agg({"usd_revenue": "sum"})
customer_revenue = customer_revenue.join(dim_customers, "customer_id")
top_customers = customer_revenue.orderBy(customer_revenue["sum(usd_revenue)"].desc()).limit(10)
display(top_customers)

In [0]:
# Top Products by Revenue – Identify products generating the highest sales revenue
product_revenue = fact_invoice_line_items.groupBy("product_id").agg({"usd_revenue": "sum"})
top_products = product_revenue.orderBy(product_revenue["sum(usd_revenue)"].desc()).limit(10)
top_products = top_products.join(dim_products, "product_id")
display(top_products)

In [0]:
# Invoice Cancellation Rate – Percentage of invoices marked as "Cancelled" vs total invoices
cancelled_count = dim_invoices.filter(dim_invoices.invoice_status == "Cancelled").count()
total_invoices = dim_invoices.count()
invoice_cancellation_rate = cancelled_count / total_invoices * 100
display(spark.createDataFrame([(invoice_cancellation_rate,)], ["Invoice Cancellation Rate (%)"]))

In [0]:
from pyspark.sql.functions import sum as _sum, avg

fact_with_customer = fact_invoice_line_items.join(
    dim_invoices.select("invoice_id", "customer_id"),
    "invoice_id"
)

discount_metrics = fact_with_customer.groupBy("customer_id").agg(
        _sum("discount").alias("total_discount"),
        avg("discount").alias("avg_discount")
    )

display(discount_metrics)

In [0]:
# Revenue by Region – Total revenue grouped by region
invoice_region = dim_invoices.select("invoice_id", "region")
revenue_by_region = fact_invoice_line_items.join(invoice_region, "invoice_id") \
    .groupBy("region").agg({"usd_revenue": "sum"})
display(revenue_by_region)

In [0]:
# Cell 11: Number of Invoices per Customer – Measures customer purchase frequency
invoices_per_customer = dim_invoices.groupBy("customer_id").agg({"invoice_id": "count"})
display(invoices_per_customer)